In [ ]:
"""
EMBEDDINGS + VECTOR DATABASE

Goal:
Convert cleaned documents into embeddings and store them
in a FAISS vector index for efficient semantic search.

Pipeline:
Clean text
    ↓
SentenceTransformer embeddings
    ↓
Normalized vectors
    ↓
FAISS vector index
    ↓
Semantic search

Improvements:
- Batch embedding generation
- Persistent embeddings
- Persistent FAISS index
- Retrieval with similarity scores
"""

In [3]:
import faiss
print("FAISS loaded successfully")

FAISS loaded successfully


In [4]:
import os
import numpy as np
import pandas as pd
import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

In [5]:
DATA_PATH = "../data/processed/newsgroups_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (19955, 3)


,doc_id,category,clean_text
0,0,alt.atheism,archive name atheism resources alt atheism arc...
1,1,alt.atheism,archive name atheism introduction alt atheism ...
2,2,alt.atheism,in article charley wingate writes the argument...
3,3,alt.atheism,until kings become philosophers or philosopher...
4,4,alt.atheism,in article bob mcgwier writes however i hate e...


In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embedding_dim = model.get_sentence_embedding_dimension()

print("Embedding dimension:", embedding_dim)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

e:\Projects\Semantic_Cache\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\saini\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


In [7]:
texts = df["clean_text"].tolist()

batch_size = 128

all_embeddings = []

for i in tqdm(range(0, len(texts), batch_size)):

    batch_texts = texts[i:i + batch_size]

    batch_embeddings = model.encode(
        batch_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    all_embeddings.append(batch_embeddings)

embeddings = np.vstack(all_embeddings)

print("Embeddings shape:", embeddings.shape)

100%|██████████| 156/156 [06:43<00:00,  2.59s/it]

Embeddings shape: (19955, 384)


In [8]:
faiss.normalize_L2(embeddings)

In [9]:
os.makedirs("../data/embeddings", exist_ok=True)

np.save("../data/embeddings/document_embeddings.npy", embeddings)

print("Embeddings saved")

Embeddings saved


In [10]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Total vectors in index:", index.ntotal)

Total vectors in index: 19955


In [11]:
faiss.write_index(index, "../data/embeddings/faiss_index.bin")

print("FAISS index saved")

FAISS index saved


In [12]:
def semantic_search(query, top_k=5):

    query_embedding = model.encode([query], convert_to_numpy=True)

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        results.append({
            "doc_id": int(df.iloc[idx]["doc_id"]),
            "category": df.iloc[idx]["category"],
            "text": df.iloc[idx]["clean_text"],
            "similarity_score": float(score)
        })

    return results

In [13]:
semantic_search("space shuttle launch")

[{'doc_id': 14881,
  'category': 'sci.space',
  'text': 'well you better not get the shuttle as your launch vehicle and most elv s have too far of a backlog for political messages if during the campaign season the candidates for president had launched one right around now we d be getting a launch for perot and if they had used the shuttle we d be seeing launches for nixon now more then ever pat',
  'similarity_score': 0.592646598815918},
 {'doc_id': 14548,
  'category': 'sci.space',
  'text': 'sorry for asking a question that s not entirely based on the technical aspects of space but i couldn t find the answer on the faqs i m currently in the uk which makes seeing a space shuttle launch a little difficult however i have been selected to be an exchange student at louisiana state uni from august and i am absolutely determined to get to see a space shuttle launch sometime during the year at which i will be in america i hear there s a bit of a long mailing list so if someone can tell me ho

In [14]:
semantic_search("baseball game season")

[{'doc_id': 10334,
  'category': 'rec.sport.hockey',
  'text': 'in article joseph charles achkar writes what did you leave the room each of the or so times they said that there were no other night baseball games every break they took back at the studio mentioned it followed by so we re gonna show you hockey instead my wife and i are hoping for rain at every baseball game they have a feed for tommorrow night point is be glad they showed hockey but if baseball was available anywhere else you can bet you would ve watched baseball last night pete clark',
  'similarity_score': 0.5556471347808838},
 {'doc_id': 9958,
  'category': 'rec.sport.baseball',
  'text': 'can someone in this net post a yankee schedule i need this right away thank you',
  'similarity_score': 0.5234315991401672},
 {'doc_id': 9651,
  'category': 'rec.sport.baseball',
  'text': 'as a philly fan as as a penna baseball fan i m anxious to see the penna series anyone know when it starts and where the first games will be playe

In [15]:
results = semantic_search("computer graphics rendering")

for r in results:
    print("\nCategory:", r["category"])
    print("Score:", r["similarity_score"])
    print("Text snippet:", r["text"][:200])


Category: comp.graphics
Score: 0.6726292967796326
Text snippet: hello i am searching for rendering software which has been developed to specifically take advantage of multi processor computer systems any pointers to such software would be greatly appreciated thank

Category: comp.graphics
Score: 0.555446982383728
Text snippet: zhenhai li writes hello i ve raytraced and rendered and the only difference i ve found is that raytracing takes a hell of a lot longer am i missing something yes there are many methods of rendering ra

Category: comp.graphics
Score: 0.548672080039978
Text snippet: technion israel institute of technology department of computer science graduate studies in computer graphics applications are invited for graduate students wishing to specialize in computer graphics a

Category: comp.graphics
Score: 0.5311852693557739
Text snippet: in article wayne michael writes corel draw will be able to do this as it will include the photopaint stuff that the pc version got with ver